# Entrega 3: Desarrollo, Entrenamiento y Experimentación del Modelo
**Proyecto:** Sistema Predictivo de Menú y Abasto de Inventario por Estaciones del Año  
**Algoritmo Principal:** Random Forest Classifier (Baseline vs. Modelo Optimizado con Poda Regularizada)  
**Fuente de Datos:** `C:\Users\villa\Downloads\dataset_limpio.csv` (149,116 registros)

In [1]:
import os
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Agregar raíz del proyecto al path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data.preprocess import DEFAULT_DATASET_PATH, cargar_datos, transformar_variables
from src.models.train import entrenar_y_evaluar
from src.models.predict import predecir

## 1. Carga de Datos desde Ruta Obligatoria y Análisis Estacional

In [2]:
# Carga obligatoria con Pandas desde C:\Users\villa\Downloads\dataset_limpio.csv
df = cargar_datos(r"C:\Users\villa\Downloads\dataset_limpio.csv")
df_proc = transformar_variables(df)
print(f"Dimensiones del dataset: {df_proc.shape}")
display(df_proc[['transaction_date', 'estacion_nombre', 'hour', 'store_location', 'product_category', 'unit_price', 'transaction_qty', 'revenue', 'beverage_temp']].head())

Dimensiones del dataset: (149116, 20)


,transaction_date,estacion_nombre,hour,store_location,product_category,unit_price,transaction_qty,revenue,beverage_temp
0,2023-01-01,Invierno,7,Lower Manhattan,Coffee,3.0,2,6.0,Hot
1,2023-01-01,Invierno,7,Lower Manhattan,Tea,3.1,2,6.2,Hot
2,2023-01-01,Invierno,7,Lower Manhattan,Drinking Chocolate,4.5,2,9.0,Hot
3,2023-01-01,Invierno,7,Lower Manhattan,Coffee,2.0,1,2.0,Hot
4,2023-01-01,Invierno,7,Lower Manhattan,Tea,3.1,2,6.2,Hot


## 2. Entrenamiento del Modelo Random Forest y Matriz de Experimentación

In [3]:
resumen = entrenar_y_evaluar(
    ruta_datos=r"C:\Users\villa\Downloads\dataset_limpio.csv",
    directorio_salida=os.path.join(PROJECT_ROOT, "models")
)
df_exp = pd.DataFrame(resumen['experiments'])
display(df_exp)

[INFO] Cargando dataset: C:\Users\villa\Downloads\dataset_limpio.csv ...
[INFO] Dataset procesado correctamente (149,116 registros).
[INFO] Conjunto Train: 119,292 muestras | Conjunto Test: 29,824 muestras
--------------------------------------------------------------------------------
[INFO] Entrenando Exp 1: Baseline - Random Forest Base (Sin Poda / Default) (n_estimators=100, max_depth=None, min_samples_split=2)...
[INFO] Exp 1: Baseline finalizado en 1.30 segundos | Train Acc: 85.36% | Test Acc: 78.31% | Test F1-Score: 0.7823
[INFO] Entrenando Exp 2: Ajuste A - Random Forest Poda Severa (Árboles Superficiales) (n_estimators=50, max_depth=4, min_samples_split=20)...
[INFO] Exp 2: Ajuste A finalizado en 0.41 segundos | Train Acc: 75.59% | Test Acc: 75.68% | Test F1-Score: 0.7382
[INFO] Entrenando Exp 3: Optimizado (Ganador) - Random Forest Optimizado (Poda Regularizada) (n_estimators=120, max_depth=12, min_samples_split=10, min_samples_leaf=4)...
[INFO] Exp 3: Optimizado (Ganador) fi

,experimento,modelo,hiperparametros,train_accuracy,test_accuracy,test_f1_weighted,test_precision_weighted,test_recall_weighted,train_time_sec,inference_time_sec,diagnostico
0,Exp 1: Baseline,Random Forest Base (Sin Poda / Default),"n_estimators=100, max_depth=None, min_samples_...",0.8536,0.7831,0.7823,0.7844,0.7831,1.30,0.49,Overfitting severo (85.36% en Train vs. 78.31%...
1,Exp 2: Ajuste A,Random Forest Poda Severa (Árboles Superficiales),"n_estimators=50, max_depth=4, min_samples_spli...",0.7559,0.7568,0.7382,0.7635,0.7568,0.41,0.19,"Underfitting (75.59% en Train, 75.68% en Test)"
2,Exp 3: Optimizado (Ganador),Random Forest Optimizado (Poda Regularizada),"n_estimators=120, max_depth=12, min_samples_sp...",0.8117,0.8048,0.8009,0.8264,0.8048,1.24,0.36,"Modelo Ganador / Óptimo (81.17% en Train, 80.4..."


## 3. Prueba Unitaria de la Función Desacoplada `predecir()`

In [4]:
muestra_invierno = {
    "estacion": "Invierno",
    "month": 1,
    "day_of_week": 1,
    "hour": 8,
    "store_id": 5,
    "unit_price": 4.25,
    "transaction_qty": 1,
    "beverage_temp": "Hot"
}
resultado = predecir(muestra_invierno)
print(json.dumps(resultado, indent=2, ensure_ascii=False))

{
  "clase_predicha": 2,
  "etiqueta": "Abasto Prioritario (Invierno): Coffee",
  "categoria_menu": "Coffee",
  "estacion_anio": "Invierno",
  "recomendacion_abasto": "Asegurar stock alto de grano tostado (Espresso, Gourmet y Drip) e insumos de vasos térmicos.",
  "top_3_categorias_probables": {
    "Coffee": 91.44,
    "Tea": 5.95,
    "Drinking Chocolate": 2.61
  },
  "confianza": 91.44,
  "status": "success"
}
